# Email Phishing Classifier — Data Generation (v2, single-source)

Phase 1 rebuild. Supersedes the v1 run for training purposes; the v1 outputs in
`data/source-clean/` stay frozen so the existing v1 Phase 2/3 models remain reproducible.

**Why v2 exists.** The v1 run merged a second Hugging Face repository,
`simlab-vs/meajor_cleaned_preprocessed` (MeAJOR). Two facts about that repository were
established by inspecting the downloaded artifact:

1. Its 108,685 rows carry only `source` values `trec5`, `trec6`, and `trec7`. It contributes
   **no Nazario Phishing Corpus and no Nigerian Fraud rows**, contrary to the v1 plan text.
   It is therefore a fully redundant, anonymized re-derivation of TREC-05/06/07, which the
   seven-corpus repository already supplies in raw form.
2. Because MeAJOR is anonymized (`[NAME]`, `[DATE]`, `[PRODUCT]`, `<|EMAIL|>`), MinHash LSH
   over word 5-gram shingles at Jaccard 0.90 could not match an anonymized copy to its raw
   twin. Deduplication and the grouped splits both missed them, so **46.5% of the v1 test set
   had a subject-twin in v1 train**, at 99.7% label agreement.

Measured effect on the v1 Phase 3 model: test FPR 1.47% on the full test set versus 2.23% on
the uncontaminated slice, and F1 0.9884 versus 0.9812. The model was real, but its published
error rates were roughly 1.5x optimistic.

**The v2 fix is to drop MeAJOR entirely**, which removes the leakage at its root and costs no
corpus family. Adding normalized subject as a split-grouping key was evaluated and rejected:
it chains transitively through sender domains into a single connected component holding 58%
of all rows, which makes an 80/10/10 split impossible. Domain-only grouping keeps the largest
group at 11.3%.

**Source (verified 2026-08-14):**
- `puyang2025/seven-phishing-email-datasets` — 203,017 rows spanning SpamAssassin,
  CEAS-08, Enron, Ling-Spam, and TREC-05/06/07.

**Safety note:** these corpora contain malicious URLs. Never fetch links or open
attachments referenced in the data.

**Outputs:** `data/source-clean-v2/{train,val,test}.parquet`, `adversarial_test.parquet`,
`manifest.json`.


## 1. Configuration

In [1]:
import hashlib, json, os
from pathlib import Path

SEED = 42
DATASET_VERSION = "v2"        # v1 outputs in data/source-clean/ are frozen, never overwritten
PROJECT_DIR = Path(os.environ.get("EMAIL_NN_DIR", "/home/spark/projects/training/email-classifier-nn"))
RAW_DIR = PROJECT_DIR / "data" / "source-raw"
CLEAN_DIR = PROJECT_DIR / "data" / f"source-clean-{DATASET_VERSION}"
RAW_DIR.mkdir(parents=True, exist_ok=True)
CLEAN_DIR.mkdir(parents=True, exist_ok=True)

HF_SEVEN = "puyang2025/seven-phishing-email-datasets"
# MeAJOR (simlab-vs/meajor_cleaned_preprocessed) is deliberately excluded in v2: it contains
# only anonymized trec5/6/7 rows, duplicating corpora the seven-corpus repository already
# provides in raw form, and its anonymization defeated near-duplicate detection.
HF_DATASETS = (HF_SEVEN,)

MIN_BODY_CHARS = 20           # quality gate: minimum stripped body length
NEAR_DUP_THRESHOLD = 0.90     # MinHash LSH Jaccard threshold
MINHASH_PERMS = 128
SHINGLE_SIZE = 5              # word shingles for near-dup detection
SPLITS = {"train": 0.80, "val": 0.10, "test": 0.10}
AUG_FRACTION = 0.15           # fraction of train phishing rows to obfuscate

CONFIG_FINGERPRINT = hashlib.sha256(json.dumps({
    "version": DATASET_VERSION, "split_alloc": "balanced-rows-label-source",
    "datasets": HF_DATASETS, "min_body": MIN_BODY_CHARS,
    "near_dup": NEAR_DUP_THRESHOLD, "perms": MINHASH_PERMS, "shingle": SHINGLE_SIZE,
    "splits": SPLITS, "aug": AUG_FRACTION, "seed": SEED,
}, sort_keys=True).encode()).hexdigest()
print("dataset version:", DATASET_VERSION)
print("output dir:", CLEAN_DIR)
print("config fingerprint:", CONFIG_FINGERPRINT[:16])


dataset version: v2
output dir: /home/spark/projects/training/email-classifier-nn/data/source-clean-v2
config fingerprint: 72487aa3531fde58


## 2. Environment

In [2]:
%pip install -q datasets pandas pyarrow datasketch

import gc, re
from datetime import datetime, timezone

import numpy as np
import pandas as pd
from datasets import load_dataset
from datasketch import MinHash, MinHashLSH

URL_RE = re.compile(r"(?:https?://|www\.)\S+", re.IGNORECASE)
WS_RE = re.compile(r"\s+")

def normalize_for_fp(subject, body):
    s = WS_RE.sub(" ", (subject or "").lower()).strip()
    b = WS_RE.sub(" ", (body or "").lower()).strip()
    return s + "\n" + b

def fingerprint(subject, body):
    return hashlib.sha256(normalize_for_fp(subject, body).encode("utf-8", "replace")).hexdigest()

def sender_domain(sender):
    if not isinstance(sender, str):
        return None
    m = re.search(r"@([A-Za-z0-9.-]+)", sender)
    return m.group(1).lower().rstrip(".") if m else None

## 3. Download Hugging Face Dataset

The repository is loaded with `datasets.load_dataset` and cached as parquet in
`data/source-raw/`; re-runs read the local parquet cache. No external corpus host or
arbitrary URL is contacted by this notebook.


In [3]:
def load_hf_cache(dataset_id, cache_path):
    if cache_path.exists():
        frame = pd.read_parquet(cache_path)
        print(f"{dataset_id} (cached): {len(frame):,} rows")
        return frame

    dataset = load_dataset(dataset_id, split="train")
    frame = dataset.to_pandas()
    tmp = Path(str(cache_path) + ".tmp")
    frame.to_parquet(tmp, index=False)
    os.replace(tmp, cache_path)
    del dataset
    gc.collect()
    print(f"{dataset_id} (downloaded): {len(frame):,} rows")
    return frame


df_seven_raw = load_hf_cache(HF_SEVEN, RAW_DIR / "seven_phishing_raw.parquet")
print(HF_SEVEN, "columns:", df_seven_raw.columns.tolist())


puyang2025/seven-phishing-email-datasets (cached): 162,413 rows
puyang2025/seven-phishing-email-datasets columns: ['text', 'subject', 'label', 'sender', 'receiver', 'date', 'urls', 'dataset_name']


## 4. Normalize

The seven-corpus schema is mapped to the shared representation kept from v1, so the v2 splits
stay column-compatible with the v1 splits and with the Phase 2/3 notebooks. `hf_dataset`
preserves repository provenance and `source` preserves the original corpus family.

`urls`, `attachment_count`, and `has_attachments` were only ever populated by MeAJOR. They are
retained as constants for schema stability; no v2 feature or model reads them.


In [4]:
COMMON_COLS = [
    "body", "subject", "sender", "receiver", "date", "urls", "url_count",
    "attachment_count", "has_attachments", "source", "label", "hf_dataset",
]

seven = df_seven_raw.rename(
    columns={"text": "body", "urls": "url_count", "dataset_name": "source"}
).copy()
seven["urls"] = ""
seven["attachment_count"] = np.nan
seven["has_attachments"] = False
seven["hf_dataset"] = HF_SEVEN

missing = set(COMMON_COLS) - set(seven.columns)
assert not missing, f"missing required columns: {sorted(missing)}"

df = seven[COMMON_COLS].copy().reset_index(drop=True)
df["body"] = df["body"].fillna("").astype(str)
df["subject"] = df["subject"].fillna("").astype(str)
df["label"] = pd.to_numeric(df["label"], errors="coerce")

assert df["hf_dataset"].nunique() == 1, "v2 must be single-source"
print(df.groupby(["hf_dataset", "source"], dropna=False).size())
print(df.groupby(["hf_dataset", "label"], dropna=False).size())
print(f"rows before quality gates: {len(df):,}")


hf_dataset                                source  
puyang2025/seven-phishing-email-datasets  Assassin     4585
                                          CEAS-08     31150
                                          Enron       23849
                                          Ling         2291
                                          TREC-05     44461
                                          TREC-06     13113
                                          TREC-07     42964
dtype: int64
hf_dataset                                label
puyang2025/seven-phishing-email-datasets  0        86951
                                          1        75462
dtype: int64
rows before quality gates: 162,413


## 5. Quality Gates

Drop rows with invalid labels or near-empty bodies.

In [5]:
n0 = len(df)
df = df[df["label"].isin([0, 1])].copy()
df["label"] = df["label"].astype(int)
df = df[df["body"].str.strip().str.len() >= MIN_BODY_CHARS].copy()
df = df.reset_index(drop=True)
print(f"quality gates: {n0:,} -> {len(df):,} rows ({n0 - len(df):,} dropped)")

quality gates: 162,413 -> 161,898 rows (515 dropped)


## 6. Deduplication

1. **Exact**: SHA-256 fingerprint of normalized `subject + body`.
2. **Near-dup**: MinHash LSH over word 5-gram shingles, clustered with union-find.
   Quoted replies/forwards in these corpora create heavy near-duplication; clusters are
   later used as split-grouping keys so near-duplicates never span splits.

This catches near-duplicates *within* the corpus. It provably could not catch v1's
cross-repository anonymized twins — which is why v2 removes that repository at the source
rather than trying to detect its rewritten copies here.

Note: building ~200k MinHash signatures is CPU-bound and can take a while.


In [6]:
df["fp"] = [fingerprint(s, b) for s, b in zip(df["subject"], df["body"])]
n0 = len(df)
df = df.drop_duplicates(subset="fp", keep="first").reset_index(drop=True)
print(f"exact dedup: {n0:,} -> {len(df):,}")

def word_shingles(subject, body, k=SHINGLE_SIZE):
    words = normalize_for_fp(subject, body).split()
    if len(words) < k:
        return {" ".join(words)} if words else set()
    return {" ".join(words[i:i + k]) for i in range(len(words) - k + 1)}

parent = list(range(len(df)))

def find(x):
    while parent[x] != x:
        parent[x] = parent[parent[x]]
        x = parent[x]
    return x

def union(a, b):
    ra, rb = find(a), find(b)
    if ra != rb:
        parent[rb] = ra

lsh = MinHashLSH(threshold=NEAR_DUP_THRESHOLD, num_perm=MINHASH_PERMS)
signatures = []
for i, (s, b) in enumerate(zip(df["subject"], df["body"])):
    mh = MinHash(num_perm=MINHASH_PERMS)
    for sh in word_shingles(s, b):
        mh.update(sh.encode("utf-8", "replace"))
    signatures.append(mh)
    lsh.insert(str(i), mh)
    if (i + 1) % 20000 == 0:
        print(f"  minhash {i + 1:,}/{len(df):,}")

for i, mh in enumerate(signatures):
    for key in lsh.query(mh):
        j = int(key)
        if j != i:
            union(i, j)

df["dup_cluster"] = [find(i) for i in range(len(df))]
print(f"near-dup clusters: {df['dup_cluster'].nunique():,} for {len(df):,} rows")
del signatures, lsh
gc.collect()

exact dedup: 161,898 -> 161,156
  minhash 20,000/161,156
  minhash 40,000/161,156
  minhash 60,000/161,156
  minhash 80,000/161,156
  minhash 100,000/161,156
  minhash 120,000/161,156
  minhash 140,000/161,156
  minhash 160,000/161,156
near-dup clusters: 153,462 for 161,156 rows


## 7. Leakage-Safe Splits

Grouping key = union of near-dup cluster and sender domain (rows sharing either are
inseparable). No group spans splits by construction.

**v2 change.** v1 assigned groups greedily by row count alone. PLAN.md specifies splits
"stratified by label and source where the grouping allows", but that stratification was never
implemented. With MeAJOR removed, domain-only grouping leaves `enron.com` as a single group
holding 11% of all rows, and row-count-only allocation produced a 44%-phishing train split
against a 69%-phishing test split with Enron at 7% of train but 43% of validation.

v2 assigns each group (largest first, seeded shuffle among ties) to whichever split minimizes
the marginal increase in squared relative deviation from target, measured jointly over row
count, both label counts, and every per-source count. Grouping is unchanged, so the leakage
guarantee is unchanged.


In [7]:
df["domain"] = df["sender"].map(sender_domain)

domain_first = {}
for i, dom in enumerate(df["domain"]):
    if isinstance(dom, str) and dom:
        if dom in domain_first:
            union(i, domain_first[dom])
        else:
            domain_first[dom] = i

df["group"] = [find(i) for i in range(len(df))]

SOURCES = sorted(df["source"].dropna().unique().tolist())
BALANCE_KEYS = ["rows", "pos", "neg"] + [f"src::{name}" for name in SOURCES]

group_stats = {}
for gid, sub in df.groupby("group"):
    stats = {
        "rows": len(sub),
        "pos": int((sub["label"] == 1).sum()),
        "neg": int((sub["label"] == 0).sum()),
    }
    source_counts = sub["source"].value_counts()
    for name in SOURCES:
        stats[f"src::{name}"] = int(source_counts.get(name, 0))
    group_stats[gid] = stats

totals = {key: sum(stats[key] for stats in group_stats.values()) for key in BALANCE_KEYS}
targets = {split: {key: SPLITS[split] * totals[key] for key in BALANCE_KEYS} for split in SPLITS}
alloc = {split: {key: 0 for key in BALANCE_KEYS} for split in SPLITS}

# largest group first; seeded shuffle breaks ties (Python sort is stable)
order = list(group_stats)
np.random.default_rng(SEED).shuffle(order)
order.sort(key=lambda gid: -group_stats[gid]["rows"])

def marginal_cost(split, stats):
    cost = 0.0
    for key in BALANCE_KEYS:
        target = targets[split][key]
        if target <= 0:
            continue
        before = alloc[split][key] - target
        after = before + stats[key]
        cost += (after * after - before * before) / target
    return cost

assign = {}
for gid in order:
    stats = group_stats[gid]
    split = min(SPLITS, key=lambda name: marginal_cost(name, stats))
    assign[gid] = split
    for key in BALANCE_KEYS:
        alloc[split][key] += stats[key]

df["split"] = df["group"].map(assign)
print(df.groupby(["split", "label"]).size().unstack(fill_value=0))
print()
print(df.groupby(["split", "source"]).size().unstack(fill_value=0))
print()
print("row share:    ", (df.groupby("split").size() / len(df)).round(4).to_dict())
print("phishing rate:", df.groupby("split")["label"].mean().round(4).to_dict())


label      0      1
split              
test    8651   7465
train  69210  59714
val     8652   7464

source  Assassin  CEAS-08  Enron  Ling  TREC-05  TREC-06  TREC-07
split                                                            
test         458     3101   2352   229     4398     1306     4272
train       3662    24808  18819  1832    35181    10446    34176
val          458     3101   2352   229     4398     1306     4272

row share:     {'test': 0.1, 'train': 0.8, 'val': 0.1}
phishing rate: {'test': 0.4632, 'train': 0.4632, 'val': 0.4631}


## 8. Feature Engineering

Per-email engineered features for the Stage-A baseline (XGBoost / small MLP):
URL stats, lexical/character statistics, obfuscation indicators.

In [8]:
SUSPICIOUS_WORDS = [
    "verify", "account", "password", "urgent", "suspend", "click",
    "confirm", "bank", "invoice", "payment", "login", "update",
]
IP_URL_RE = re.compile(r"https?://\d{1,3}(?:\.\d{1,3}){3}", re.IGNORECASE)
HTML_TAG_RE = re.compile(r"</?[a-zA-Z][^>]*>")
ZW_CHARS = "\u200b\u200c\u200d\ufeff"

def engineer(row):
    body, subject = row["body"], row["subject"]
    urls = URL_RE.findall(body)
    low = (subject + " " + body).lower()
    blen = max(len(body), 1)
    return {
        "f_body_len": len(body),
        "f_subject_len": len(subject),
        "f_url_count": len(urls),
        "f_url_ip": sum(bool(IP_URL_RE.match(u)) for u in urls),
        "f_url_at": sum("@" in u for u in urls),
        "f_upper_ratio": sum(c.isupper() for c in body) / blen,
        "f_digit_ratio": sum(c.isdigit() for c in body) / blen,
        "f_exclaim": body.count("!"),
        "f_html_tags": len(HTML_TAG_RE.findall(body)),
        "f_susp_words": sum(w in low for w in SUSPICIOUS_WORDS),
        "f_nonascii_ratio": sum(ord(c) > 127 for c in body) / blen,
        "f_zero_width": sum(c in ZW_CHARS for c in body),
    }

features = pd.DataFrame([engineer(r) for _, r in df.iterrows()])
FEATURE_COLS = features.columns.tolist()
df = pd.concat([df.reset_index(drop=True), features], axis=1)
print(df[FEATURE_COLS].describe().T[["mean", "std", "max"]])

                         mean           std           max
f_body_len        1778.790098  11056.275661  2.550619e+06
f_subject_len       36.166553     31.234330  7.170000e+03
f_url_count          1.510344     10.741501  3.134000e+03
f_url_ip             0.003500      0.120603  1.800000e+01
f_url_at             0.036852      0.316509  2.700000e+01
f_upper_ratio        0.052998      0.067800  9.859155e-01
f_digit_ratio        0.031046      0.041058  8.518519e-01
f_exclaim            1.316116      9.481165  1.879000e+03
f_html_tags          0.011852      1.905049  4.660000e+02
f_susp_words         0.412129      0.820371  1.200000e+01
f_nonascii_ratio     0.006308      0.058001  1.000000e+00
f_zero_width         0.000012      0.003523  1.000000e+00


## 9. Adversarial Augmentation (train only)

Deterministic obfuscation copies of a seeded sample of **train** phishing rows:
Cyrillic homoglyph substitution + zero-width character injection into trigger words.
A separate fixed adversarial eval set is built from **test** phishing rows (originals
stay in test; the obfuscated copies are saved to `adversarial_test.parquet` only).
Engineered features are recomputed after obfuscation.

In [9]:
HOMOGLYPHS = {
    "a": "\u0430", "c": "\u0441", "e": "\u0435", "i": "\u0456",
    "o": "\u043e", "p": "\u0440", "s": "\u0455", "x": "\u0445",
}
ZW = "\u200b"
TRIGGER_RE = re.compile(
    r"verify|account|password|urgent|bank|login|click|confirm", re.IGNORECASE)

def obfuscate(text, seed):
    r = np.random.default_rng(seed)
    out = []
    for ch in text:
        sub = HOMOGLYPHS.get(ch.lower())
        if sub and r.random() < 0.30:
            out.append(sub.upper() if ch.isupper() else sub)
        else:
            out.append(ch)
    s = "".join(out)
    return TRIGGER_RE.sub(lambda m: m.group(0)[:2] + ZW + m.group(0)[2:], s)

df["augmented"] = False

def make_adversarial(rows, seed_base):
    adv = rows.copy()
    adv["body"] = [obfuscate(b, seed_base + i) for i, b in enumerate(adv["body"])]
    adv["subject"] = [obfuscate(s, seed_base + 500000 + i) for i, s in enumerate(adv["subject"])]
    adv["augmented"] = True
    refreshed = pd.DataFrame([engineer(r) for _, r in adv.iterrows()], index=adv.index)
    adv[FEATURE_COLS] = refreshed
    return adv

train_phish = df[(df["split"] == "train") & (df["label"] == 1)]
train_aug = make_adversarial(train_phish.sample(frac=AUG_FRACTION, random_state=SEED), SEED)

test_phish = df[(df["split"] == "test") & (df["label"] == 1)]
adv_eval = make_adversarial(test_phish, SEED + 1000000)

print(f"train augmentations: {len(train_aug):,}")
print(f"adversarial eval set: {len(adv_eval):,}")

train augmentations: 8,957
adversarial eval set: 7,465


## 10. Verify & Save

Leakage assertions (no shared group or fingerprint across splits), the v2 subject-twin
regression check, class/source distribution report, atomic parquet writes, and a
fingerprinted manifest.


In [10]:
for a, b in [("train", "val"), ("train", "test"), ("val", "test")]:
    ga = set(df.loc[df["split"] == a, "group"])
    gb = set(df.loc[df["split"] == b, "group"])
    assert not (ga & gb), f"group leakage between {a} and {b}"
    fa = set(df.loc[df["split"] == a, "fp"])
    fb = set(df.loc[df["split"] == b, "fp"])
    assert not (fa & fb), f"fingerprint leakage between {a} and {b}"
print("leakage checks passed")

# v2 regression check for the defect that motivated this rebuild. In v1 this rate was 46.5%
# on test. It cannot reach zero here: distinct messages in one mailing-list thread legitimately
# share a subject, and domain-only grouping does not separate them. It is recorded in the
# manifest so any future regression is visible rather than silent.
RE_PREFIX_RE = re.compile(r"^((re|fwd|fw|aw|sv)\s*:\s*)+", re.IGNORECASE)

def norm_subject(value):
    return WS_RE.sub(" ", RE_PREFIX_RE.sub("", str(value or "").lower())).strip()

df["norm_subject"] = df["subject"].map(norm_subject)
long_subj = df[df["norm_subject"].str.len() >= 15]
train_subjects = set(long_subj.loc[long_subj["split"] == "train", "norm_subject"])
subject_twin_rate = {}
for holdout in ("val", "test"):
    rows = long_subj[long_subj["split"] == holdout]
    shared = rows["norm_subject"].isin(train_subjects)
    rate = float(shared.mean()) if len(rows) else 0.0
    subject_twin_rate[holdout] = rate
    print(f"{holdout}: subject-twin-in-train = {rate:.1%} ({int(shared.sum()):,}/{len(rows):,})")

INTERNAL_COLS = ["fp", "dup_cluster", "group", "domain", "split", "norm_subject"]

def save_atomic(frame, path):
    tmp = Path(str(path) + ".tmp")
    serializable = frame.drop(columns=INTERNAL_COLS, errors="ignore").copy()
    serializable["date"] = serializable["date"].astype("string")
    serializable.to_parquet(tmp, index=False)
    os.replace(tmp, path)

outputs = {
    "train": pd.concat([df[df["split"] == "train"], train_aug], ignore_index=True)
               .sample(frac=1.0, random_state=SEED).reset_index(drop=True),
    "val": df[df["split"] == "val"].reset_index(drop=True),
    "test": df[df["split"] == "test"].reset_index(drop=True),
    "adversarial_test": adv_eval.reset_index(drop=True),
}
counts = {}
for name, frame in outputs.items():
    path = CLEAN_DIR / f"{name}.parquet"
    save_atomic(frame, path)
    counts[name] = {
        "rows": int(len(frame)),
        "phish": int(frame["label"].sum()),
        "legit": int((frame["label"] == 0).sum()),
        "augmented": int(frame["augmented"].sum()),
    }
    print(f"{name}: {counts[name]}")

manifest = {
    "dataset_version": DATASET_VERSION,
    "config_fingerprint": CONFIG_FINGERPRINT,
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "sources": list(HF_DATASETS),
    "excluded_sources": {
        "simlab-vs/meajor_cleaned_preprocessed":
            "trec5/6/7 only; anonymized duplicate of corpora already present raw; "
            "defeated MinHash near-dup detection and caused 46.5% v1 test contamination",
    },
    "feature_columns": FEATURE_COLS,
    "subject_twin_rate": subject_twin_rate,
    "counts": counts,
}
tmp = CLEAN_DIR / "manifest.json.tmp"
tmp.write_text(json.dumps(manifest, indent=2))
os.replace(tmp, CLEAN_DIR / "manifest.json")
print("manifest written:", CLEAN_DIR / "manifest.json")


leakage checks passed
val: subject-twin-in-train = 34.9% (5,047/14,466)
test: subject-twin-in-train = 29.7% (4,145/13,938)
train: {'rows': 137881, 'phish': 68671, 'legit': 69210, 'augmented': 8957}
val: {'rows': 16116, 'phish': 7464, 'legit': 8652, 'augmented': 0}
test: {'rows': 16116, 'phish': 7465, 'legit': 8651, 'augmented': 0}
adversarial_test: {'rows': 7465, 'phish': 7465, 'legit': 0, 'augmented': 7465}
manifest written: /home/spark/projects/training/email-classifier-nn/data/source-clean-v2/manifest.json
